# 🚀 Lakeview Dashboard Migration (Parameterized)

This notebook migrates a Lakeview dashboard from one Databricks workspace to another with full support for Unity Catalog volumes, automatic catalog/schema mapping, and permissions migration.

## Key Features
- ✅ Read dashboard JSON from Unity Catalog volumes
- ✅ Automatic catalog/schema mapping via lookup file
- ✅ Verify source location and validate .lvdash.json files
- ✅ Cross-reference table names with lookup file
- ✅ Save updated JSON to target volume
- ✅ **Migrate dashboard permissions (ACLs)** - NEW!
- ✅ Support for PAT and OAuth authentication

## Migration Steps
1. **Verify Source Location** - Check dashboard JSON exists in source volume
2. **Extract References** - List all catalog.schema.table references
3. **Load Lookup File** - Cross-reference with mapping file
4. **Rewrite References** - Update all catalog/schema references
5. **Save to Target Volume** - Store updated JSON in target volume
6. **Validate Queries** - Test queries against target workspace
7. **Import Dashboard** - Create dashboard in target workspace
7.5. **Migrate Permissions** - Copy ACLs from source to target (optional)
8. **Publish Dashboard** - Make dashboard accessible
9. **Summary** - Review migration results

## 📋 Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - Update these values for your migration
# ============================================================================

# Source Configuration
SOURCE_WORKSPACE_URL = "https://e2-demo-field-eng.cloud.databricks.com"
SOURCE_CATALOG = "demo_catalog"
SOURCE_SCHEMA = "demo_schema"
SOURCE_VOLUME = "migration_files"
SOURCE_DASHBOARD_FILENAME = "category_insights_dashboard.lvdash.json"  # Dashboard JSON file in volume

# Source volume path (constructed from above)
SOURCE_VOLUME_PATH = f"/Volumes/{SOURCE_CATALOG}/{SOURCE_SCHEMA}/{SOURCE_VOLUME}/{SOURCE_DASHBOARD_FILENAME}"

# Target Configuration
TARGET_WORKSPACE_URL = "https://adb-7405609619727450.10.azuredatabricks.net"
TARGET_CATALOG = "archana_krish_fe_dsa"
TARGET_SCHEMA = "vizient_deep_dive"
TARGET_VOLUME = "migration_files"
TARGET_DASHBOARD_FILENAME = "dashboard_updated.lvdash.json"  # Output filename in target volume

# Target volume path (constructed from above)
TARGET_VOLUME_PATH = f"/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/{TARGET_VOLUME}/{TARGET_DASHBOARD_FILENAME}"

# Target Dashboard Configuration
TARGET_DASHBOARD_NAME = "Category Insights - Healthcare Supply Chain"
TARGET_DASHBOARD_PATH = "/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration"
TARGET_WAREHOUSE_ID = None  # ⚠️ SET THIS: Get from SQL Warehouses in target workspace

# Lookup File Configuration
# This file contains catalog/schema/table mappings: old_catalog, old_schema, old_table, new_catalog, new_schema, new_table
LOOKUP_FILE_NAME = "catalog_mapping.csv"  # CSV file in same directory as this notebook
LOOKUP_FILE_PATH = f"{LOOKUP_FILE_NAME}"  # Or full path: "/Workspace/Users/.../catalog_mapping.csv"

# Alternative: Use JSON lookup file format
# LOOKUP_FILE_NAME = "catalog_mapping.json"
# Format: {"old_catalog.old_schema": "new_catalog.new_schema"}

# Authentication Configuration
AUTH_METHOD = "PAT"  # "PAT" or "OAUTH"

# PAT tokens (use dbutils.secrets or environment variables in production)
SOURCE_PAT_TOKEN = None  # Or: dbutils.secrets.get(scope="my-scope", key="source_pat")
TARGET_PAT_TOKEN = None  # Or: dbutils.secrets.get(scope="my-scope", key="target_pat")

# OAuth Configuration (if AUTH_METHOD == "OAUTH")
# Set environment variables: ARM_CLIENT_ID, ARM_TENANT_ID, ARM_CLIENT_SECRET

# Migration Options
VALIDATE_QUERIES = True  # Validate queries against target workspace before import
CREATE_BACKUP = True  # Create backup in target volume before replacing

# Dashboard Import Method: "programmatic" or "manual"
IMPORT_METHOD = "manual"  # Set to "programmatic" for automated import, "manual" to do it yourself

# Publish Options (only applies if IMPORT_METHOD = "programmatic")
PUBLISH_DASHBOARD = True  # Automatically publish dashboard after programmatic import

# Permissions Configuration (REQUIRED for this workflow)
# Source dashboard ID is needed to migrate permissions
SOURCE_DASHBOARD_ID = None  # ⚠️ SET THIS: Get from source dashboard URL

# Map users/groups from source to target workspace (if names differ)
USER_GROUP_MAPPING = {
    # Example mappings if user/group names differ between workspaces:
    # "source_user@example.com": "target_user@example.com",
    # "source_group": "target_group"
}
PERMISSIONS_BACKUP_FILE = "dashboard_permissions_backup.json"  # Save permissions to file

print("=" * 80)
print("📋 CONFIGURATION SUMMARY")
print("=" * 80)
print(f"Source Volume: {SOURCE_VOLUME_PATH}")
print(f"Target Volume: {TARGET_VOLUME_PATH}")
print(f"Lookup File: {LOOKUP_FILE_PATH}")
print(f"Auth Method: {AUTH_METHOD}")
print(f"Import Method: {IMPORT_METHOD}")
print(f"Source Dashboard ID: {SOURCE_DASHBOARD_ID or '⚠️  NOT SET'}")
print("=" * 80)
print("\n💡 Authentication is used for:")
print("   ✅ Reading dashboard JSON from source volume")
print("   ✅ Writing updated JSON to target volume")
print("   ✅ Migrating permissions (source → target)")
print(f"   {'✅' if IMPORT_METHOD == 'programmatic' else '⚠️ '} Dashboard import ({'programmatic' if IMPORT_METHOD == 'programmatic' else 'manual via UI'})")
print(f"   {'✅' if VALIDATE_QUERIES else '⚠️ '} Query validation ({'enabled' if VALIDATE_QUERIES else 'disabled'})")
print("=" * 80)
print("✅ Configuration loaded")

## 🔧 Setup and Dependencies

In [ ]:
# Install required packages if not already installed
import subprocess
import sys

def install_package(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_package("databricks-sdk")
install_package("requests")

print("✅ Dependencies installed")

## 🔐 Authentication Setup

In [ ]:
import os
import json
import re
import time
from datetime import datetime
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import Dashboard
import requests

# Initialize workspace clients based on authentication method
def create_workspace_client(workspace_url, token=None):
    """Create a WorkspaceClient with PAT or OAuth authentication."""
    if AUTH_METHOD == "PAT":
        if not token:
            raise ValueError("PAT token is required when AUTH_METHOD is 'PAT'")
        return WorkspaceClient(
            host=workspace_url,
            token=token
        )
    else:  # OAUTH
        # OAuth uses environment variables: ARM_CLIENT_ID, ARM_TENANT_ID, ARM_CLIENT_SECRET
        # Or Azure CLI credentials (az login)
        return WorkspaceClient(
            host=workspace_url
        )

# Create clients
try:
    source_client = create_workspace_client(SOURCE_WORKSPACE_URL, SOURCE_PAT_TOKEN)
    target_client = create_workspace_client(TARGET_WORKSPACE_URL, TARGET_PAT_TOKEN)
    print("✅ Workspace clients created successfully")
    print(f"   Source: {SOURCE_WORKSPACE_URL}")
    print(f"   Target: {TARGET_WORKSPACE_URL}")
except Exception as e:
    print(f"❌ Error creating workspace clients: {e}")
    print("\n💡 Troubleshooting:")
    print("   - For PAT: Ensure SOURCE_PAT_TOKEN and TARGET_PAT_TOKEN are set")
    print("   - For OAuth: Ensure ARM_CLIENT_ID, ARM_TENANT_ID, ARM_CLIENT_SECRET are set as env vars")
    raise

## 📍 Step 1: Verify Source Location and Load Dashboard JSON

### Verify and Load Dashboard JSON from Volume

In [ ]:
# ============================================================================
# STEP 1: Verify Source Location and Load Dashboard JSON
# ============================================================================

import pandas as pd

print(f"📍 Step 1: Verifying source location...")
print(f"   Source Volume Path: {SOURCE_VOLUME_PATH}\n")

# Check if file exists
try:
    # Try to read the file from volume
    with open(SOURCE_VOLUME_PATH, 'r', encoding='utf-8') as f:
        dashboard_json = json.load(f)
    
    print(f"✅ Dashboard JSON found and loaded successfully!")
    print(f"   File: {SOURCE_DASHBOARD_FILENAME}")
    print(f"   Location: {SOURCE_VOLUME_PATH}")
    
    # Validate it's a proper .lvdash.json file
    if not SOURCE_DASHBOARD_FILENAME.endswith('.lvdash.json'):
        print(f"⚠️  Warning: File does not have .lvdash.json extension")
    
    # Check for required fields
    required_fields = ['pages', 'datasets']
    missing_fields = [field for field in required_fields if field not in dashboard_json]
    
    if missing_fields:
        print(f"⚠️  Warning: Missing required fields: {missing_fields}")
    else:
        print(f"✅ Dashboard structure validated")
    
    # Display dashboard info
    if 'display_name' in dashboard_json:
        print(f"   Dashboard Name: {dashboard_json['display_name']}")
    
    if 'datasets' in dashboard_json:
        print(f"   Datasets: {len(dashboard_json['datasets'])}")
    
    if 'pages' in dashboard_json:
        print(f"   Pages: {len(dashboard_json['pages'])}")
    
    # Create serialized version for later use
    serialized_dashboard = json.dumps(dashboard_json, separators=(",", ":"), ensure_ascii=False)
    
except FileNotFoundError:
    print(f"❌ Error: Dashboard JSON file not found")
    print(f"   Expected location: {SOURCE_VOLUME_PATH}")
    print(f"\n💡 Troubleshooting:")
    print(f"   1. Verify the file exists in the volume")
    print(f"   2. Check catalog, schema, and volume names")
    print(f"   3. Ensure you have READ permissions on the volume")
    print(f"   4. Use dbfs ls command: dbutils.fs.ls('{SOURCE_VOLUME_PATH}')")
    raise

except json.JSONDecodeError as e:
    print(f"❌ Error: Invalid JSON format in dashboard file")
    print(f"   Error: {e}")
    raise

except Exception as e:
    print(f"❌ Error loading dashboard JSON: {e}")
    raise

## 🔍 Step 2: Extract and List All Catalog.Schema.Table References

In [ ]:
# ============================================================================
# STEP 2: Extract All Catalog.Schema.Table References
# ============================================================================

def discover_catalog_schema_references(dashboard_json):
    """Discover all catalog.schema.table references in dashboard queries."""
    references = set()
    
    def extract_references(obj, path=""):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if isinstance(value, str):
                    # Check if this is a SQL query or expression field
                    if key in ("query", "sql", "statement") or "query" in key.lower() or "expression" in key.lower():
                        # Extract catalog.schema.table patterns
                        # Pattern: catalog.schema.table or `catalog`.`schema`.`table`
                        patterns = [
                            r"`([a-zA-Z0-9_]+)`\.`([a-zA-Z0-9_]+)`\.`([a-zA-Z0-9_]+)`",
                            r"\b([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)\b"
                        ]
                        for pattern in patterns:
                            matches = re.findall(pattern, value)
                            for match in matches:
                                if len(match) == 3:
                                    references.add((match[0], match[1], match[2]))
                else:
                    extract_references(value, f"{path}.{key}")
        elif isinstance(obj, list):
            for i, item in enumerate(obj):
                extract_references(item, f"{path}[{i}]")
    
    extract_references(dashboard_json)
    return sorted(references)

print(f"📊 Step 2: Extracting catalog.schema.table references...\n")

# Discover references
references = discover_catalog_schema_references(dashboard_json)

print(f"🔍 Found {len(references)} unique catalog.schema.table references:\n")

if references:
    # Create DataFrame for better visualization
    ref_df = pd.DataFrame(references, columns=['Catalog', 'Schema', 'Table'])
    ref_df['Full_Reference'] = ref_df.apply(lambda row: f"{row['Catalog']}.{row['Schema']}.{row['Table']}", axis=1)
    
    print(ref_df.to_string(index=False))
    
    # Group by catalog and schema
    print(f"\n📈 Summary by Catalog:")
    catalog_counts = ref_df.groupby('Catalog').size().reset_index(name='Count')
    print(catalog_counts.to_string(index=False))
    
    print(f"\n📈 Summary by Schema:")
    schema_counts = ref_df.groupby(['Catalog', 'Schema']).size().reset_index(name='Count')
    print(schema_counts.to_string(index=False))
else:
    print("⚠️  No catalog.schema.table references found in dashboard JSON")
    print("   The dashboard might use views, temporary tables, or different reference patterns")

## 📋 Step 3: Load Lookup File and Cross-Reference

In [ ]:
# ============================================================================
# STEP 3: Load Lookup File and Cross-Reference
# ============================================================================

print(f"📋 Step 3: Loading lookup file...\n")
print(f"   Lookup File: {LOOKUP_FILE_PATH}\n")

def load_lookup_file(file_path):
    """Load catalog/schema/table mapping from CSV or JSON file."""
    try:
        if file_path.endswith('.csv'):
            # Load CSV format
            # Expected columns: old_catalog, old_schema, old_table, new_catalog, new_schema, new_table
            lookup_df = pd.read_csv(file_path)
            
            # Validate required columns
            required_cols = ['old_catalog', 'old_schema', 'old_table', 'new_catalog', 'new_schema', 'new_table']
            missing_cols = [col for col in required_cols if col not in lookup_df.columns]
            
            if missing_cols:
                print(f"❌ Error: Lookup file missing required columns: {missing_cols}")
                print(f"   Required columns: {required_cols}")
                return None
            
            return lookup_df
            
        elif file_path.endswith('.json'):
            # Load JSON format
            # Expected format: {"old_catalog.old_schema": "new_catalog.new_schema"}
            with open(file_path, 'r') as f:
                lookup_dict = json.load(f)
            
            # Convert to DataFrame format
            rows = []
            for old_ref, new_ref in lookup_dict.items():
                old_parts = old_ref.split('.')
                new_parts = new_ref.split('.')
                
                if len(old_parts) >= 2 and len(new_parts) >= 2:
                    rows.append({
                        'old_catalog': old_parts[0],
                        'old_schema': old_parts[1],
                        'old_table': old_parts[2] if len(old_parts) > 2 else '*',  # Wildcard for all tables
                        'new_catalog': new_parts[0],
                        'new_schema': new_parts[1],
                        'new_table': new_parts[2] if len(new_parts) > 2 else '*'
                    })
            
            return pd.DataFrame(rows)
        else:
            print(f"❌ Error: Unsupported file format. Use .csv or .json")
            return None
            
    except FileNotFoundError:
        print(f"❌ Error: Lookup file not found: {file_path}")
        print(f"\n💡 Create a lookup file with one of these formats:")
        print(f"\n   CSV format (catalog_mapping.csv):")
        print(f"   old_catalog,old_schema,old_table,new_catalog,new_schema,new_table")
        print(f"   demo_catalog,demo_schema,table1,prod_catalog,prod_schema,table1")
        print(f"   demo_catalog,demo_schema,*,prod_catalog,prod_schema,*")
        print(f"\n   JSON format (catalog_mapping.json):")
        print(f"   {{")
        print(f"     \"demo_catalog.demo_schema\": \"prod_catalog.prod_schema\"")
        print(f"   }}")
        return None
    except Exception as e:
        print(f"❌ Error loading lookup file: {e}")
        return None

# Load lookup file
lookup_df = load_lookup_file(LOOKUP_FILE_PATH)

if lookup_df is not None and not lookup_df.empty:
    print(f"✅ Lookup file loaded successfully")
    print(f"   Mappings found: {len(lookup_df)}\n")
    print("📋 Lookup Mappings:")
    print(lookup_df.to_string(index=False))
    
    # Cross-reference with discovered references
    print(f"\n🔗 Cross-Referencing with Dashboard References:\n")
    
    matched_refs = []
    unmatched_refs = []
    
    for catalog, schema, table in references:
        # Try to find exact match
        exact_match = lookup_df[
            (lookup_df['old_catalog'] == catalog) &
            (lookup_df['old_schema'] == schema) &
            (lookup_df['old_table'] == table)
        ]
        
        # Try wildcard match (table = '*')
        wildcard_match = lookup_df[
            (lookup_df['old_catalog'] == catalog) &
            (lookup_df['old_schema'] == schema) &
            (lookup_df['old_table'] == '*')
        ]
        
        if not exact_match.empty:
            match_row = exact_match.iloc[0]
            matched_refs.append({
                'old_ref': f"{catalog}.{schema}.{table}",
                'new_ref': f"{match_row['new_catalog']}.{match_row['new_schema']}.{match_row['new_table']}",
                'match_type': 'exact'
            })
        elif not wildcard_match.empty:
            match_row = wildcard_match.iloc[0]
            matched_refs.append({
                'old_ref': f"{catalog}.{schema}.{table}",
                'new_ref': f"{match_row['new_catalog']}.{match_row['new_schema']}.{table}",  # Keep original table name
                'match_type': 'wildcard'
            })
        else:
            unmatched_refs.append(f"{catalog}.{schema}.{table}")
    
    # Display results
    if matched_refs:
        matched_df = pd.DataFrame(matched_refs)
        print(f"✅ Matched References ({len(matched_refs)}):")
        print(matched_df.to_string(index=False))
    
    if unmatched_refs:
        print(f"\n⚠️  Unmatched References ({len(unmatched_refs)}):")
        for ref in unmatched_refs:
            print(f"   - {ref}")
        print(f"\n💡 Add these to your lookup file if they need to be mapped")
    
    if not matched_refs and not unmatched_refs:
        print("ℹ️  No references to map")
        
else:
    print("⚠️  No lookup file loaded. Will skip catalog/schema rewriting.")
    matched_refs = []
    lookup_df = pd.DataFrame()

## 🔄 Step 4: Rewrite Catalog/Schema References

In [ ]:
# ============================================================================
# STEP 4: Rewrite Catalog/Schema References
# ============================================================================

print(f"🔄 Step 4: Rewriting catalog/schema references...\n")

def rewrite_sql_references(sql_string, lookup_df):
    """Rewrite catalog.schema.table references in SQL strings using lookup table."""
    if lookup_df is None or lookup_df.empty:
        return sql_string
    
    rewritten = sql_string
    
    # Apply each mapping
    for _, row in lookup_df.iterrows():
        old_catalog = row['old_catalog']
        old_schema = row['old_schema']
        old_table = row['old_table']
        new_catalog = row['new_catalog']
        new_schema = row['new_schema']
        new_table = row['new_table']
        
        if old_table == '*':
            # Wildcard mapping - replace catalog.schema only
            # Pattern 1: Backtick-quoted identifiers
            pattern1 = rf"`{re.escape(old_catalog)}`\.`{re.escape(old_schema)}`\."
            replacement1 = f"`{new_catalog}`.`{new_schema}`."
            rewritten = re.sub(pattern1, replacement1, rewritten, flags=re.IGNORECASE)
            
            # Pattern 2: Bare identifiers
            pattern2 = rf"\b{re.escape(old_catalog)}\.{re.escape(old_schema)}\."
            replacement2 = f"{new_catalog}.{new_schema}."
            rewritten = re.sub(pattern2, replacement2, rewritten, flags=re.IGNORECASE)
        else:
            # Exact table mapping
            # Pattern 1: Backtick-quoted full reference
            pattern1 = rf"`{re.escape(old_catalog)}`\.`{re.escape(old_schema)}`\.`{re.escape(old_table)}`"
            replacement1 = f"`{new_catalog}`.`{new_schema}`.`{new_table}`"
            rewritten = re.sub(pattern1, replacement1, rewritten, flags=re.IGNORECASE)
            
            # Pattern 2: Bare full reference
            pattern2 = rf"\b{re.escape(old_catalog)}\.{re.escape(old_schema)}\.{re.escape(old_table)}\b"
            replacement2 = f"{new_catalog}.{new_schema}.{new_table}"
            rewritten = re.sub(pattern2, replacement2, rewritten, flags=re.IGNORECASE)
    
    return rewritten


def rewrite_dashboard_json(dashboard_json, lookup_df):
    """Recursively rewrite catalog/schema references in dashboard JSON."""
    if lookup_df is None or lookup_df.empty:
        return dashboard_json
    
    def walk_and_rewrite(obj):
        if isinstance(obj, dict):
            result = {}
            for key, value in obj.items():
                # Check if this is a SQL query or expression field
                if isinstance(value, str) and (
                    key in ("query", "sql", "statement") or 
                    "query" in key.lower() or 
                    "expression" in key.lower()
                ):
                    result[key] = rewrite_sql_references(value, lookup_df)
                else:
                    result[key] = walk_and_rewrite(value)
            return result
        elif isinstance(obj, list):
            return [walk_and_rewrite(item) for item in obj]
        else:
            return obj
    
    return walk_and_rewrite(dashboard_json)


# Rewrite references if lookup file was loaded
if lookup_df is not None and not lookup_df.empty:
    print(f"   Applying {len(lookup_df)} mapping(s)...\n")
    
    # Create a copy for comparison
    original_serialized = json.dumps(dashboard_json, separators=(",", ":"), ensure_ascii=False)
    
    # Rewrite
    rewritten_json = rewrite_dashboard_json(dashboard_json, lookup_df)
    updated_serialized = json.dumps(rewritten_json, separators=(",", ":"), ensure_ascii=False)
    
    # Compare
    if original_serialized != updated_serialized:
        print(f"✅ References rewritten successfully!")
        
        # Count changes
        total_changes = 0
        for _, row in lookup_df.iterrows():
            old_pattern = f"{row['old_catalog']}.{row['old_schema']}"
            if row['old_table'] != '*':
                old_pattern += f".{row['old_table']}"
            total_changes += original_serialized.count(old_pattern)
        
        print(f"   Changed {total_changes} reference(s)")
        
        # Verify new references
        new_references = discover_catalog_schema_references(rewritten_json)
        print(f"\n📊 New References ({len(new_references)}):")
        
        if new_references:
            new_ref_df = pd.DataFrame(new_references, columns=['Catalog', 'Schema', 'Table'])
            new_ref_df['Full_Reference'] = new_ref_df.apply(
                lambda row: f"{row['Catalog']}.{row['Schema']}.{row['Table']}", axis=1
            )
            print(new_ref_df.to_string(index=False))
        
        # Update dashboard JSON
        dashboard_json = rewritten_json
        serialized_dashboard = updated_serialized
        
    else:
        print("ℹ️  No changes made (references may already be correct)")
else:
    print("ℹ️  No lookup file loaded. Skipping rewrite step.")
    print("   Dashboard JSON will be used as-is.")

## 💾 Step 5: Save Updated JSON to Target Volume

In [ ]:
# ============================================================================
# STEP 5: Save Updated JSON to Target Volume
# ============================================================================

print(f"💾 Step 5: Saving updated dashboard JSON...\n")
print(f"   Target Volume Path: {TARGET_VOLUME_PATH}\n")

try:
    # Create backup if requested and file already exists
    if CREATE_BACKUP:
        try:
            with open(TARGET_VOLUME_PATH, 'r') as f:
                existing_content = f.read()
            
            # Create backup filename with timestamp
            backup_filename = TARGET_DASHBOARD_FILENAME.replace('.lvdash.json', f'_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.lvdash.json')
            backup_path = TARGET_VOLUME_PATH.replace(TARGET_DASHBOARD_FILENAME, backup_filename)
            
            with open(backup_path, 'w', encoding='utf-8') as f:
                f.write(existing_content)
            
            print(f"💾 Backup created: {backup_filename}")
        except FileNotFoundError:
            print(f"ℹ️  No existing file to backup")
    
    # Ensure target volume directory exists (create if needed)
    target_dir = os.path.dirname(TARGET_VOLUME_PATH)
    os.makedirs(target_dir, exist_ok=True)
    
    # Save updated JSON
    updated_json_str = json.dumps(dashboard_json, indent=2, ensure_ascii=False)
    
    with open(TARGET_VOLUME_PATH, 'w', encoding='utf-8') as f:
        f.write(updated_json_str)
    
    print(f"✅ Dashboard JSON saved successfully!")
    print(f"   Location: {TARGET_VOLUME_PATH}")
    print(f"   File size: {len(updated_json_str):,} bytes")
    print(f"   Catalog: {TARGET_CATALOG}")
    print(f"   Schema: {TARGET_SCHEMA}")
    print(f"   Volume: {TARGET_VOLUME}")
    
    # Update serialized version for import
    serialized_dashboard = json.dumps(dashboard_json, separators=(",", ":"), ensure_ascii=False)
    
except PermissionError:
    print(f"❌ Error: Permission denied writing to target volume")
    print(f"\n💡 Troubleshooting:")
    print(f"   1. Verify you have WRITE permissions on the volume")
    print(f"   2. Check volume access grants: GRANT WRITE ON VOLUME {TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_VOLUME} TO `user@example.com`")
    print(f"   3. Ensure the volume exists: CREATE VOLUME IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_VOLUME}")
    raise

except Exception as e:
    print(f"❌ Error saving to target volume: {e}")
    raise

## ✅ Step 6: Validate Queries (Optional)

In [ ]:
# ============================================================================
# STEP 6: Validate Queries Against Target Workspace (Optional)
# ============================================================================

print(f"✅ Step 6: Validating queries...\n")

def extract_dataset_queries(dashboard_json):
    """Extract all dataset queries from dashboard JSON."""
    queries = []
    
    if "datasets" in dashboard_json:
        for dataset in dashboard_json["datasets"]:
            if "query" in dataset:
                queries.append({
                    "name": dataset.get("name", "unknown"),
                    "query": dataset["query"]
                })
    
    return queries


def validate_query(client, warehouse_id, query_sql, catalog=None, schema=None):
    """Validate a query using Statement Execution API."""
    try:
        # Use SQL execution API
        statement = client.statement_execution.execute_statement(
            warehouse_id=warehouse_id,
            statement=query_sql,
            catalog=catalog,
            schema=schema
        )
        
        # Wait for completion
        statement_id = statement.statement_id
        max_wait = 30  # seconds
        waited = 0
        
        while waited < max_wait:
            status = client.statement_execution.get_statement(statement_id)
            state = status.status.state
            
            if state in ("SUCCEEDED", "FAILED", "CANCELED"):
                return {
                    "success": state == "SUCCEEDED",
                    "state": state,
                    "message": getattr(status.status, 'message', None)
                }
            
            time.sleep(1)
            waited += 1
        
        return {
            "success": False,
            "state": "TIMEOUT",
            "message": f"Query validation timed out after {max_wait} seconds"
        }
        
    except Exception as e:
        return {
            "success": False,
            "state": "ERROR",
            "message": str(e)
        }


# Validate queries if requested
if VALIDATE_QUERIES and TARGET_WAREHOUSE_ID:
    print("   Extracting queries from dashboard...")
    queries = extract_dataset_queries(dashboard_json)
    print(f"   Found {len(queries)} query/queries\n")
    
    validation_results = []
    for query_info in queries:
        print(f"   Validating: {query_info['name']}...", end=" ")
        result = validate_query(
            target_client,
            TARGET_WAREHOUSE_ID,
            query_info['query'],
            catalog=TARGET_CATALOG,
            schema=TARGET_SCHEMA
        )
        
        validation_results.append({
            "name": query_info['name'],
            **result
        })
        
        status_icon = "✅" if result["success"] else "❌"
        print(f"{status_icon} {result['state']}")
        if not result["success"] and result["message"]:
            print(f"      Error: {result['message'][:100]}")
    
    # Summary
    successful = sum(1 for r in validation_results if r["success"])
    total = len(validation_results)
    print(f"\n📊 Validation Summary: {successful}/{total} queries passed")
    
    if successful < total:
        print("⚠️  Some queries failed validation. Review errors above.")
        print("   You may need to update your catalog/schema mappings or fix query syntax")
    else:
        print("✅ All queries validated successfully!")
else:
    print("ℹ️  Query validation skipped")
    print("   Set VALIDATE_QUERIES=True and TARGET_WAREHOUSE_ID to enable")

## 📥 Step 7: Import Dashboard to Target Workspace (Programmatic or Manual)

### Choose Your Import Method

You have two options for importing the dashboard:

**Option A: Manual Import (Recommended for first-time users)**
1. Navigate to target workspace: `{TARGET_WORKSPACE_URL}`
2. Go to **SQL** > **Dashboards**
3. Click **"Create"** > **"Import dashboard"**
4. Upload the updated `.lvdash.json` file from: `{TARGET_VOLUME_PATH}`
5. Select target SQL Warehouse
6. Click **"Import"** → Note the dashboard ID from the URL

**Option B: Programmatic Import (Automated)**
- Set `IMPORT_METHOD = "programmatic"` in configuration
- Requires `TARGET_WAREHOUSE_ID` to be set
- Automatically creates dashboard and returns ID

---

### Programmatic Import (if enabled)

In [ ]:
# ============================================================================
# STEP 7: Import Dashboard to Target Workspace
# ============================================================================

print(f"📥 Step 7: Importing dashboard...\n")
print(f"   Import Method: {IMPORT_METHOD.upper()}\n")

def import_dashboard_programmatic(client, dashboard_name, dashboard_path, serialized_dashboard, warehouse_id):
    """Import dashboard using Lakeview API."""
    try:
        # Create Dashboard object
        dashboard = Dashboard.from_dict({
            "display_name": dashboard_name,
            "parent_path": dashboard_path,
            "warehouse_id": warehouse_id,
            "serialized_dashboard": serialized_dashboard
        })
        
        # Create dashboard
        created_dashboard = client.lakeview.create(dashboard=dashboard)
        
        return created_dashboard.dashboard_id
    except Exception as e:
        print(f"❌ Error importing dashboard: {e}")
        raise


# Import dashboard based on method
if IMPORT_METHOD == "programmatic":
    # Programmatic import
    if not TARGET_WAREHOUSE_ID:
        print("❌ Error: TARGET_WAREHOUSE_ID not set for programmatic import")
        print("\n💡 To find your warehouse ID:")
        print("   1. Go to SQL Warehouses in target workspace")
        print("   2. Select your warehouse")
        print("   3. Go to Connection details")
        print("   4. Copy the Warehouse ID (e.g., 'abc123def456')")
        print("\n   Then set: TARGET_WAREHOUSE_ID = 'your-warehouse-id' in configuration cell")
        print("\n⚠️  Or change IMPORT_METHOD = 'manual' to import via UI")
        raise ValueError("TARGET_WAREHOUSE_ID required for programmatic import")
    
    print(f"   🤖 Programmatic Import Starting...")
    print(f"   Target Workspace: {TARGET_WORKSPACE_URL}")
    print(f"   Dashboard Name: {TARGET_DASHBOARD_NAME}")
    print(f"   Dashboard Path: {TARGET_DASHBOARD_PATH}")
    print(f"   Warehouse ID: {TARGET_WAREHOUSE_ID}\n")
    
    target_dashboard_id = import_dashboard_programmatic(
        target_client,
        TARGET_DASHBOARD_NAME,
        TARGET_DASHBOARD_PATH,
        serialized_dashboard,
        TARGET_WAREHOUSE_ID
    )
    
    print(f"\n✅ Dashboard imported programmatically!")
    print(f"   Dashboard ID: {target_dashboard_id}")
    print(f"   Dashboard URL: {TARGET_WORKSPACE_URL}/sql/dashboardsv3/{target_dashboard_id}")
    
elif IMPORT_METHOD == "manual":
    # Manual import instructions
    print(f"   👤 Manual Import Required")
    print(f"\n   📋 Follow these steps:")
    print(f"   1. Navigate to: {TARGET_WORKSPACE_URL}")
    print(f"   2. Go to: SQL > Dashboards")
    print(f"   3. Click: Create > Import dashboard")
    print(f"   4. Upload file from: {TARGET_VOLUME_PATH}")
    print(f"   5. Select SQL Warehouse (any warehouse in target workspace)")
    print(f"   6. Click: Import")
    print(f"   7. ⚠️  IMPORTANT: Copy the dashboard ID from the URL")
    print(f"      URL format: .../sql/dashboardsv3/[DASHBOARD_ID]")
    print(f"\n   8. Set the dashboard ID below and run the next cell:")
    print(f"      target_dashboard_id = 'your-copied-dashboard-id'")
    
    # Prompt user to set the dashboard ID
    print(f"\n   ⏸️  Paused: Waiting for manual import...")
    print(f"   After importing, set target_dashboard_id in the cell below")
    
else:
    print(f"❌ Error: Invalid IMPORT_METHOD '{IMPORT_METHOD}'")
    print(f"   Valid options: 'programmatic' or 'manual'")
    raise ValueError(f"Invalid IMPORT_METHOD: {IMPORT_METHOD}")

## 📝 Step 7a: Set Target Dashboard ID (Manual Import Only)

**If you did MANUAL import:** Set the dashboard ID you copied from the URL below.

**If you did PROGRAMMATIC import:** Skip this cell - the ID is already set.

In [ ]:
# ============================================================================
# Set Target Dashboard ID (MANUAL IMPORT ONLY)
# ============================================================================

# If you did MANUAL import, set the dashboard ID here:
# You can find this from the dashboard URL after import:
# Example URL: https://workspace.databricks.com/sql/dashboardsv3/abc123def456
#              The ID is: abc123def456

# Uncomment and set if you did manual import:
# target_dashboard_id = "abc123def456"  # Replace with your actual dashboard ID

if 'target_dashboard_id' in locals():
    print(f"✅ Target dashboard ID set: {target_dashboard_id}")
    print(f"   Dashboard URL: {TARGET_WORKSPACE_URL}/sql/dashboardsv3/{target_dashboard_id}")
else:
    if IMPORT_METHOD == "manual":
        print("⚠️  Target dashboard ID not set")
        print("   Uncomment and set: target_dashboard_id = 'your-id-here'")
    else:
        print("ℹ️  Programmatic import - dashboard ID already set")

## 🔐 Step 7.5: Migrate Dashboard Permissions (REQUIRED)

**Why this step is needed:**
Dashboard permissions control who can view/edit the dashboard in the target workspace.

**What gets migrated:**
- User permissions (CAN_VIEW, CAN_RUN, CAN_EDIT, CAN_MANAGE)
- Group permissions
- Service principal permissions

**Prerequisites:**
- `SOURCE_DASHBOARD_ID` must be set in configuration (Cell 2)
- `target_dashboard_id` must be set (from Step 7 or 7a)
- Authentication tokens must have permission to read/write ACLs

**What happens:**
1. Export permissions from source dashboard
2. Map users/groups if configured
3. Save backup to `dashboard_permissions_backup.json`
4. Apply permissions to target dashboard

In [ ]:
# ============================================================================
# STEP 7.5: Migrate Dashboard Permissions
# ============================================================================

print(f"🔐 Step 7.5: Migrating dashboard permissions...\n")

def get_dashboard_permissions(client, workspace_url, dashboard_id):
    """
    Get permissions (ACLs) for a Lakeview dashboard.
    Note: Requires proper access to source dashboard.
    """
    try:
        # Lakeview dashboards use workspace permissions API
        # The path format is /sql/dashboards/{dashboard_id}
        object_id = f"/sql/dashboards/{dashboard_id}"
        
        # Make direct API call to get permissions
        import requests
        
        # Extract host from workspace_url
        host = workspace_url.rstrip('/')
        token = None
        
        # Get token based on auth method
        if AUTH_METHOD == "PAT":
            token = SOURCE_PAT_TOKEN if client == source_client else TARGET_PAT_TOKEN
        else:
            # For OAuth, get token from client config
            token = client.config.token
        
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        # Get permissions using Workspace API
        url = f"{host}/api/2.0/permissions/dashboards/{dashboard_id}"
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 404:
            print(f"   ⚠️  Dashboard permissions endpoint not found (dashboard may be private)")
            return None
        else:
            print(f"   ⚠️  Could not retrieve permissions: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"   ⚠️  Error getting permissions: {e}")
        return None


def map_principal(principal, mapping):
    """Map user/group names from source to target workspace."""
    if principal in mapping:
        return mapping[principal]
    return principal


def set_dashboard_permissions(client, workspace_url, dashboard_id, acl_list):
    """
    Set permissions (ACLs) for a Lakeview dashboard.
    """
    try:
        import requests
        
        # Get token based on auth method
        token = None
        if AUTH_METHOD == "PAT":
            token = TARGET_PAT_TOKEN
        else:
            token = client.config.token
        
        host = workspace_url.rstrip('/')
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        # Set permissions using Workspace API
        url = f"{host}/api/2.0/permissions/dashboards/{dashboard_id}"
        
        payload = {
            "access_control_list": acl_list
        }
        
        response = requests.put(url, headers=headers, json=payload)
        
        if response.status_code in (200, 201):
            return True
        else:
            print(f"   ⚠️  Could not set permissions: {response.status_code}")
            print(f"   Response: {response.text}")
            return False
            
    except Exception as e:
        print(f"   ❌ Error setting permissions: {e}")
        return False


# Migrate permissions (REQUIRED STEP)
if 'target_dashboard_id' not in locals():
    print("❌ Error: target_dashboard_id not set")
    print("   Cannot migrate permissions without target dashboard ID")
    print("\n💡 If you did manual import:")
    print("   Go back to Step 7a and set: target_dashboard_id = 'your-id'")
    print("\n💡 If you did programmatic import:")
    print("   Check Step 7 for errors - dashboard may not have been created")
    raise ValueError("target_dashboard_id is required for permissions migration")

print("   Starting permissions migration...\n")

# Get source dashboard ID
source_dashboard_id = None

# Check if set in configuration
if SOURCE_DASHBOARD_ID:
    source_dashboard_id = SOURCE_DASHBOARD_ID
    print(f"   ℹ️  Using SOURCE_DASHBOARD_ID from configuration")
# Check if we have a dashboard_id field in the loaded JSON
elif 'dashboard_id' in dashboard_json:
    source_dashboard_id = dashboard_json.get('dashboard_id')
    print(f"   ℹ️  Found dashboard_id in JSON file")

# Validate source dashboard ID
if not source_dashboard_id:
    print("   ❌ Error: Source dashboard ID not found")
    print("   💡 Set SOURCE_DASHBOARD_ID in Configuration (Cell 2)")
    print("      Get it from source dashboard URL:")
    print(f"      Example: {SOURCE_WORKSPACE_URL}/sql/dashboardsv3/[DASHBOARD_ID]")
    raise ValueError("SOURCE_DASHBOARD_ID is required for permissions migration")
else:
        print(f"   Source Dashboard ID: {source_dashboard_id}")
        print(f"   Target Dashboard ID: {target_dashboard_id}\n")
        
        # Get source permissions
        source_permissions = get_dashboard_permissions(
            source_client, 
            SOURCE_WORKSPACE_URL, 
            source_dashboard_id
        )
        
        if source_permissions and 'access_control_list' in source_permissions:
            acl_list = source_permissions['access_control_list']
            
            print(f"   ✅ Found {len(acl_list)} permission(s) in source dashboard\n")
            
            # Display source permissions
            print("   📋 Source Permissions:")
            for acl in acl_list:
                principal = acl.get('user_name') or acl.get('group_name') or acl.get('service_principal_name', 'unknown')
                permissions = acl.get('all_permissions', [])
                perm_level = permissions[0].get('permission_level', 'unknown') if permissions else 'unknown'
                print(f"      - {principal}: {perm_level}")
            
            # Map principals if needed
            mapped_acl_list = []
            for acl in acl_list:
                new_acl = acl.copy()
                
                # Map user names
                if 'user_name' in acl:
                    new_acl['user_name'] = map_principal(acl['user_name'], USER_GROUP_MAPPING)
                
                # Map group names
                if 'group_name' in acl:
                    new_acl['group_name'] = map_principal(acl['group_name'], USER_GROUP_MAPPING)
                
                mapped_acl_list.append(new_acl)
            
            # Save permissions to backup file
            backup_path = PERMISSIONS_BACKUP_FILE
            with open(backup_path, 'w', encoding='utf-8') as f:
                json.dump({
                    "source_dashboard_id": source_dashboard_id,
                    "target_dashboard_id": target_dashboard_id,
                    "source_permissions": source_permissions,
                    "timestamp": datetime.now().isoformat()
                }, f, indent=2, ensure_ascii=False)
            
            print(f"\n   💾 Permissions backed up to: {backup_path}")
            
            # Apply permissions to target dashboard
            print(f"\n   🔄 Applying permissions to target dashboard...")
            success = set_dashboard_permissions(
                target_client,
                TARGET_WORKSPACE_URL,
                target_dashboard_id,
                mapped_acl_list
            )
            
            if success:
                print(f"   ✅ Permissions migrated successfully!")
                print(f"\n   📋 Target Permissions:")
                for acl in mapped_acl_list:
                    principal = acl.get('user_name') or acl.get('group_name') or acl.get('service_principal_name', 'unknown')
                    permissions = acl.get('all_permissions', [])
                    perm_level = permissions[0].get('permission_level', 'unknown') if permissions else 'unknown'
                    print(f"      - {principal}: {perm_level}")
            else:
                print(f"   ⚠️  Could not apply permissions automatically")
                print(f"   💡 You may need to set permissions manually in the target workspace")
                print(f"      Or check that users/groups exist in target workspace")
        else:
            print("   ⚠️  Could not apply permissions automatically")
            print("   💡 You may need to set permissions manually in the target workspace")
            print("      Or check that users/groups exist in target workspace")
            print("\n   📋 Manual Permission Setup:")
            print("      1. Open target dashboard in UI")
            print("      2. Click 'Share' button")
            print("      3. Add users/groups with appropriate permission levels")
            print("      4. Available levels: CAN_VIEW, CAN_RUN, CAN_EDIT, CAN_MANAGE")
    else:
        print("   ℹ️  No permissions found or could not access source permissions")
        print("   💡 This is normal if:")
        print("      - Source dashboard is private")
        print("      - You don't have admin access to source workspace")
        print("      - Dashboard uses default permissions only")
        print("\n   📋 Set permissions manually:")
        print("      1. Open target dashboard in UI")
        print("      2. Click 'Share' button")
        print("      3. Add users/groups with appropriate permission levels")
        print("      4. Available levels: CAN_VIEW, CAN_RUN, CAN_EDIT, CAN_MANAGE")

## 📢 Step 8: Publish Dashboard (Optional - Programmatic Import Only)

In [ ]:
# ============================================================================
# STEP 8: Publish Dashboard (Programmatic Import Only)
# ============================================================================

print(f"📢 Step 8: Publishing dashboard...\n")

if IMPORT_METHOD == "manual":
    print("ℹ️  Manual import - publishing not applicable")
    print("   Manually imported dashboards are published via UI:")
    print("   1. Open dashboard in target workspace")
    print("   2. Click 'Publish' button if available")
    print("   3. Or dashboard may already be in published state")
    
elif IMPORT_METHOD == "programmatic":
    def publish_dashboard(client, dashboard_id, embed_credentials=False):
        """Publish dashboard."""
        try:
            client.lakeview.publish(
                dashboard_id=dashboard_id,
                embed_credentials=embed_credentials
            )
            return True
        except Exception as e:
            print(f"❌ Error publishing dashboard: {e}")
            return False
    
    # Publish dashboard if requested
    if PUBLISH_DASHBOARD and 'target_dashboard_id' in locals():
        print(f"   Publishing programmatically...")
        print(f"   Dashboard ID: {target_dashboard_id}")
        print(f"   Embed Credentials: False\n")
        
        success = publish_dashboard(target_client, target_dashboard_id, embed_credentials=False)
        
        if success:
            print(f"✅ Dashboard published successfully!")
            print(f"   Published URL: {TARGET_WORKSPACE_URL}/sql/dashboardsv3/{target_dashboard_id}")
            print(f"\n💡 The dashboard is now accessible to users with appropriate permissions")
        else:
            print(f"⚠️  Failed to publish dashboard")
            print(f"   You can publish manually from the dashboard UI")
    else:
        print("ℹ️  Publishing skipped")
        if not PUBLISH_DASHBOARD:
            print("   Set PUBLISH_DASHBOARD=True to auto-publish")
        if 'target_dashboard_id' not in locals():
            print("   Dashboard was not imported successfully")

## ✅ Step 9: Migration Summary

In [ ]:
print("\n" + "=" * 80)
print("📊 MIGRATION SUMMARY")
print("=" * 80)

print(f"\n📍 Source:")
print(f"   Workspace: {SOURCE_WORKSPACE_URL}")
print(f"   Volume: {SOURCE_VOLUME_PATH}")
print(f"   Catalog: {SOURCE_CATALOG}")
print(f"   Schema: {SOURCE_SCHEMA}")
print(f"   Dashboard File: {SOURCE_DASHBOARD_FILENAME}")

print(f"\n📍 Target:")
print(f"   Workspace: {TARGET_WORKSPACE_URL}")
print(f"   Volume: {TARGET_VOLUME_PATH}")
print(f"   Catalog: {TARGET_CATALOG}")
print(f"   Schema: {TARGET_SCHEMA}")
print(f"   Dashboard File: {TARGET_DASHBOARD_FILENAME}")

if 'target_dashboard_id' in locals():
    print(f"\n✅ Migrated Dashboard:")
    print(f"   Dashboard ID: {target_dashboard_id}")
    print(f"   Dashboard Name: {TARGET_DASHBOARD_NAME}")
    print(f"   Dashboard Path: {TARGET_DASHBOARD_PATH}")
    print(f"   Dashboard URL: {TARGET_WORKSPACE_URL}/sql/dashboardsv3/{target_dashboard_id}")
    print(f"   Status: {'Published' if PUBLISH_DASHBOARD else 'Draft'}")
else:
    print(f"\n⚠️  Dashboard not imported")
    print(f"   Check errors above or verify TARGET_WAREHOUSE_ID is set")

if 'matched_refs' in locals() and matched_refs:
    print(f"\n🔄 Catalog/Schema Mappings Applied: {len(matched_refs)}")

if 'lookup_df' in locals() and lookup_df is not None and not lookup_df.empty:
    print(f"📋 Lookup File Used: {LOOKUP_FILE_PATH} ({len(lookup_df)} mappings)")

print("\n" + "=" * 80)
print("🎉 MIGRATION COMPLETED SUCCESSFULLY!")
print("=" * 80)

print(f"\n💡 Next Steps:")
print(f"   1. Open dashboard: {TARGET_WORKSPACE_URL}/sql/dashboardsv3/{target_dashboard_id if 'target_dashboard_id' in locals() else '[ID]'}")
print(f"   2. Verify all visuals load correctly")
print(f"   3. Test filters and interactions")
print(f"   4. Check data accuracy from new catalog/schema")
print(f"   5. Share with stakeholders")